In [ ]:
!pip install -U bitsandbytes transformers accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 38.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 145.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 536.6/536.6 kB 51.3 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface-hub 0.36.0
    Uninstalling huggingface-hub-0.36.0:
      Successfully uninstalled huggingface-hub-0.36.0
  Attempting uninstall: transformers
    Found existing installation: transformers 4.57.6
    Uninstalling transformers-4.57.6:
      Successfully uninstalled transformers-4.57.6


## Load Nested JSON Data

### Subtask:
Load the data from the nested JSON file, which contains objects with 'id', 'syllogism', 'validity', and 'plausibility' fields.


In [ ]:
import json
import pandas as pd

# Use the correct relative path from the notebook's location to the data file
with open('train_data.json', 'r') as f:
    data = json.load(f)

df_syllogisms = pd.DataFrame(data)

print("DataFrame loaded successfully. Displaying the first 5 rows:")
print(df_syllogisms.head())

syllogism_sentences = df_syllogisms['syllogism'].tolist()

print(f"Extracted {len(syllogism_sentences)} syllogism sentences. Displaying the first 3:")
for i, sentence in enumerate(syllogism_sentences[:3]):
    print(f"- {sentence}")

DataFrame loaded successfully. Displaying the first 5 rows:
                                     id  \
0  50146f21-d265-4e3a-8d93-8165cdbe89a3   
1  dfafb4f6-4e1d-4cd5-aeb4-75d36aafdf1a   
2  e30b1f83-a4c3-49cb-8aaf-5f64208c625b   
3  a30e07d5-0fb3-4097-9892-4b145b0c54f5   
4  5b8161b7-b1bf-4e16-a854-cd52cdce8a1b   

                                           syllogism  validity  plausibility  
0  All cars are a type of vehicle. No animal is a...     False          True  
1  Nothing that is a soda is a juice. A portion o...      True          True  
2  Everything that is a planet is a celestial bod...     False         False  
3  Every cat is an invisible creature. A number o...      True         False  
4  There are no capital cities which are oceans. ...      True          True  
Extracted 960 syllogism sentences. Displaying the first 3:
- All cars are a type of vehicle. No animal is a car. Therefore, no animal can be a vehicle.
- Nothing that is a soda is a juice. A portion of the t

# LLAMA Experiment

## Map concepts to placeholders



In [ ]:
def map_concepts_to_placeholders(concepts):
    """
    Maps a list of three concepts to logical placeholders 'X', 'Y', 'Z'.
    Assumes the input list contains exactly three concepts.
    """
    if len(concepts) != 3:
        raise ValueError("Expected exactly three concepts for mapping to X, Y, Z.")

    concept_to_placeholder = {
        concepts[0]: 'X',
        concepts[1]: 'Y',
        concepts[2]: 'Z'
    }
    return concept_to_placeholder

# Test the function with the refined concepts from the previous step
# mapping_1 = map_concepts_to_placeholders(refined_concepts_1)
# mapping_2 = map_concepts_to_placeholders(refined_concepts_2)
# mapping_3 = map_concepts_to_placeholders(refined_concepts_3)

# print(f"Mapping for Syllogism 1 ({refined_concepts_1}): {mapping_1}")
# print(f"Mapping for Syllogism 2 ({refined_concepts_2}): {mapping_2}")
# print(f"Mapping for Syllogism 3 ({refined_concepts_3}): {mapping_3}")

In [ ]:
# Check GPU availability and configure PyTorch
import torch

print("=" * 80)
print("GPU CONFIGURATION")
print("=" * 80)
print(f"GPU Available: {torch.cuda.is_available()}")
print(f"GPU Count: {torch.cuda.device_count()}")

if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"GPU Total Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
    print(f"GPU Current Device: {torch.cuda.current_device()}")
else:
    print("WARNING: No GPU detected. Model will run on CPU (slow).")
print("=" * 80)

GPU CONFIGURATION
GPU Available: True
GPU Count: 1
GPU Name: NVIDIA A100-SXM4-40GB
GPU Total Memory: 42.47 GB
GPU Current Device: 0


In [ ]:
import torch
torch.cuda.empty_cache()

In [ ]:
# Load Llama-3.2-1B for intelligent concept extraction
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
import re


llama_model_name = "meta-llama/Llama-3.1-8B-Instruct"
print(f"Loading {llama_model_name} for concept extraction...")

# Use use_auth_token and trust_remote_code=True so the remote repo's custom class (e.g. LlamaForCausalLM) can be loaded
llama_tokenizer = AutoTokenizer.from_pretrained(
    llama_model_name
    )

from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch

# 1. Setup 8-bit configuration
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4"
)

if torch.cuda.is_available():
    print(f"Loading {llama_model_name} on GPU...")
    llama_model = AutoModelForCausalLM.from_pretrained(
        llama_model_name,
        quantization_config=bnb_config, # Modern way to do load_in_8bit
        device_map="auto"
    )
    print(f"Model loaded on GPU: {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: Loading on CPU. This will be very slow.")
    llama_model = AutoModelForCausalLM.from_pretrained(
        llama_model_name,
        torch_dtype=torch.float32,
        device_map="cpu"
    )

print(f"{llama_model_name} model loaded successfully.")

Loading meta-llama/Llama-3.1-8B-Instruct for concept extraction...


config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/55.4k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

...

model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

Model loaded on GPU: NVIDIA A100-SXM4-40GB
meta-llama/Llama-3.1-8B-Instruct model loaded successfully.


In [ ]:
def greedy_unify_concepts(concepts):
    """
    If one concept is a substring of another, keeps the longer one.
    Example: ['invisible', 'cat', 'invisible creature'] -> ['invisible creature', 'cat']
    """
    # Sort by length descending so we check longest phrases first
    sorted_concepts = sorted(list(set(concepts)), key=len, reverse=True)
    unified = []

    for concept in sorted_concepts:
        # Check if this concept is already covered by a longer phrase in our unified list
        if not any(concept in existing for existing in unified):
            unified.append(concept)
    return unified

In [ ]:
import json
import re

def parse_json(text):
    # 1. Find the first '{'
    start_idx = text.find('{')
    if start_idx == -1:
        return None

    # 2. Isolate the FIRST balanced object
    brace_count = 0
    potential_json = None
    for i in range(start_idx, len(text)):
        if text[i] == '{':
            brace_count += 1
        elif text[i] == '}':
            brace_count -= 1

        if brace_count == 0:
            potential_json = text[start_idx:i+1]
            break # Stop immediately after the first complete object

    if not potential_json:
        return None

    # 3. FIX: Handle unquoted values (Common Llama-3.2-3B error)
    # This regex looks for values not starting with " or [ or { and wraps them in quotes
    try:
        return json.loads(potential_json)
    except json.JSONDecodeError:
        try:
            # Matches : followed by text that doesn't start with a quote,
            # and captures everything until the comma or closing brace
            fixed = re.sub(r':\s*([^"\[\{].*?)(?=\s*[,\}])', r': "\1"', potential_json)
            return json.loads(fixed)
        except:
            return None

In [ ]:
def extract_concepts_llm(sentence):
    # We increase max_new_tokens so the LLM has room to 'think'
    # Then we use a marker like "FINAL CONCEPTS:" to grab the result

    sentences = sentence.split('.')

    prompt = f"""
  Task: You will be given 3 sentences that form a syllogism. Extract the 3 core concepts from the syllogism.

  Rules:
  1. Every syllogism has 3 concepts.
  2. Each concept MUST appear in exactly 2 different sentences.
  3. Each sentence MUST contain exactly 2 concepts.
  3. NEVER include quantifiers such as: all, no, some, every in the concepts.

  Follow this step-by-step reasoning:
  Step 1: List the nouns/noun phrases in each sentence.
  Step 2: Identify the key concepts from each sentence (Example: Aristotle is a man -> key concepts: Aristotle, man)
  Step 3: Identify which 3 concepts appear in 2 different sentences.
  Step 4: Output the results

  Output rules:
  - Output only the reasoning and the extracted concepts
  - Do not output any additional information such as notes or examples
  - Do not output code
  - Do not output any additional JSONs other than the one detailed in Output structure

  Output structure:
  Resoning: <your reasoning>,
  {{
      "concepts": ["concept 1", "concept 2", "concept 3"]
  }}

  Sentence 1: {sentences[0]}
  Sentence 2: {sentences[1]}
  Sentence 3: {sentences[2]}

  OUTPUT:
    """

    inputs = llama_tokenizer(prompt, return_tensors="pt").to(llama_model.device)

    with torch.no_grad():
        outputs = llama_model.generate(
            **inputs,
            max_new_tokens=512,  # Increased to allow for Chain of Thought
            temperature=0,
            do_sample=False,
            eos_token_id=llama_tokenizer.eos_token_id,
        )

    response = llama_tokenizer.decode(outputs[0], skip_special_tokens=True)
    generated_text = response[len(prompt):].strip()
    print(f"Output:\n{generated_text}")

    # --- Post-Processing for 1B/3B Models ---

    generated_json = parse_json(generated_text)
    if generated_json and "concepts" in generated_json:
      final_concepts = generated_json["concepts"]
    else:
      # Fallback or error handling
      final_concepts = ["error_1", "error_2", "error_3"]

    # 5. Ensure we return exactly 3
    while len(final_concepts) < 3:
        final_concepts.append(f'unknown_{len(final_concepts) + 1}')

    return final_concepts[:3]

## Replace concepts with placeholders




In [ ]:
def replace_concepts_with_placeholders(sentence, concept_mapping):
    """
    Replaces identified LLM-extracted concepts with placeholders (X, Y, Z).
    Works with Llama-extracted concepts directly.
    Preserves quantifiers and logical structure.
    """
    result = sentence

    # Sort concepts by length (longest first) to avoid partial replacements
    sorted_concepts = sorted(concept_mapping.items(), key=lambda x: len(x[0]), reverse=True)

    # Replace each concept with its placeholder, preserving quantifiers
    for concept, placeholder in sorted_concepts:
        # Case-insensitive replacement with word boundaries
        # Pattern: match concept with optional preceding quantifiers
        pattern = rf'\b({concept})\b'

        # Function to preserve preceding quantifiers
        def replace_func(match):
            return placeholder

        result = re.sub(pattern, replace_func, result, flags=re.IGNORECASE)

    return result

In [ ]:
import re

def expand_concept_mapping(concept_mapping):
    """
    Expands the mapping to include unique parts of the concepts.
    If 'invisible creature' is mapped to 'X', and 'invisible' helps distinguish it
    from the other two concepts, we map 'invisible' to 'X' as well.
    """
    # 1. Invert map to get list of concepts: {'X': 'invisible creature', ...}
    placeholder_to_concept = {v: k for k, v in concept_mapping.items()}
    all_concepts = list(concept_mapping.keys())

    # 2. Tokenize all concepts into sets of words
    # We ignore small words like 'of', 'a', 'in' to be safe
    def get_tokens(text):
        return set(w.lower() for w in re.findall(r'\b\w{3,}\b', text))

    concept_tokens = {c: get_tokens(c) for c in all_concepts}

    # 3. Find unique tokens for each concept
    expanded_mapping = concept_mapping.copy()

    for concept, tokens in concept_tokens.items():
        current_placeholder = concept_mapping[concept]

        for token in tokens:
            # Check if this token appears in ANY other concept
            is_unique = True
            for other_concept, other_tokens in concept_tokens.items():
                if other_concept != concept and token in other_tokens:
                    is_unique = False
                    break

            # If the token is unique to this concept (e.g., "invisible"),
            # map the token directly to the placeholder (e.g., "invisible" -> "X")
            if is_unique:
                expanded_mapping[token] = current_placeholder

    return expanded_mapping

def replace_concepts_with_placeholders(sentence, concept_mapping):
    """
    Updated function that uses the expanded mapping to catch partial matches.
    """
    # 1. Expand the mapping to catch variations (e.g. 'invisible')
    full_mapping = expand_concept_mapping(concept_mapping)

    result = sentence

    # 2. Sort by length (longest first) is CRITICAL here.
    # We want to replace "invisible creature" (len 18) before "invisible" (len 9)
    sorted_concepts = sorted(full_mapping.items(), key=lambda x: len(x[0]), reverse=True)

    # 3. Perform replacement
    for concept_phrase, placeholder in sorted_concepts:
        # We look for the phrase as a whole word (\b)
        pattern = rf'\b{re.escape(concept_phrase)}\b'

        # We replace it with the placeholder
        # Note: We do NOT need to handle quantifiers here; we just want to swap
        # the noun phrase. The quantifier stays in the text (e.g., "All X").
        result = re.sub(pattern, placeholder, result, flags=re.IGNORECASE)

    return result

In [ ]:
# DEBUG concept mapping
def process_syllogism_with_llm(sentence, use_llm=True):
    """
    Complete pipeline: extract concepts (LLM or spaCy), map to placeholders, and transform.
    """
    sentence = lemmatize_sentence(sentence)
    print(f"Lemmatized sentece: {sentence}")

    # Step 1: Extract concepts
    concepts = extract_concepts_llm(sentence)

    # Step 2: Map concepts to placeholders
    mapping = map_concepts_to_placeholders(concepts)

    # Step 3: Replace concepts with placeholders
    transformed = replace_concepts_with_placeholders(sentence, mapping)

    return concepts, mapping, transformed


# Test the updated pipeline with LLM concepts
# print("=" * 80)
# print("UPDATED PIPELINE TEST: Using Llama-3.2-1B for Concept Extraction")
# print("=" * 80)

# test_indices = [1]

# for idx in test_indices:
#     sentence = syllogism_sentences[idx]
#     print(f"\n{'='*80}")
#     print(f"Syllogism {idx}: {sentence}")
#     print(f"{'='*80}")

#     # Process with LLM
#     concepts, mapping, transformed = process_syllogism_with_llm(sentence, use_llm=True)

#     print(f"LLM Extracted Concepts: {concepts}")
#     print(f"Concept→Placeholder Mapping: {mapping}")
#     print(f"Transformed (placeholders): {transformed}")


## Enhance Rule-Based Simplification



In [ ]:
import re

def simplify_syllogism(text):
    """
    Simplifies a syllogism to a standard structure with bidirectional variable mapping.
    """
    # 1. Extract original variables (X, Y, or Z) in order of appearance
    original_matches = re.findall(r'\b[XYZ]\b', text)
    unique_vars = []
    for v in original_matches:
        if v not in unique_vars:
            unique_vars.append(v)

    # 2. Pre-process: Map original variables to X and Y for the model
    if len(unique_vars) >= 2:
        # Create mapping (e.g., {'Y': 'X', 'Z': 'Y'})
        input_mapping = {unique_vars[0]: 'X', unique_vars[1]: 'Y'}

        # Use a function to replace both simultaneously (prevents variable collision)
        def map_to_model(match):
            return input_mapping.get(match.group(0), match.group(0))

        normalized_sentence = re.sub(r'\b[XYZ]\b', map_to_model, text)
    else:
        normalized_sentence = text

    # 3. Prompt Construction
    prompt = f"""You are an expert in syllogism that classifies sentences into one of three logic structures. You will recieve a sentence and will output which category it belongs to.

Categories:
- All X are Y.
- Some X are Y.
- Some X are not Y.
- No X are Y.

Classification rules:
- Ignore filler words, focus only on the subjects and their quantity (all, some, none)
- The negation of 'All' is 'Some' (e.g. Not all X are Y -> Some X are Y)
- The absence of quantity == All (e.g. X are Y -> All X are Y)
- Some X are not Y == Some X are Y

Follow this step-by-step reasoning:
Step 1: Identify the subjects (X and Y)
Step 2: Identify the subects' quantifiers and transform them into 'All', 'Some' or 'No'
Step 3: Identify the filler words and remove them
Step 4: Output the results

Output Rules:
- Output ONLY your reasoning and the category
- DO NOT output any extra text

Output structure:
Resoning: <your reasoning>,
{{
    "category": <identified category>
}}

Sentence: {normalized_sentence}
Output:"""

    # 4. Model Inference
    inputs = llama_tokenizer(prompt, return_tensors="pt").to(llama_model.device)

    with torch.no_grad():
        outputs = llama_model.generate(
            **inputs,
            max_new_tokens=96,
            temperature=0.0,
            do_sample=False,
            eos_token_id=llama_tokenizer.eos_token_id,
        )

    response = llama_tokenizer.decode(outputs[0], skip_special_tokens=True)

    # Extract generated part and clean leading dashes/spaces
    generated_text = response[len(prompt):].strip()
    # print(f"genrated_text\n{generated_text}")

    generated_json = parse_json(generated_text)
    # print(f"genrated_json\n{generated_json}")

    if generated_json and "category" in generated_json:
      simplified = generated_json["category"]
    else:
      # Fallback or error handling
      simplified = "Error"

    if simplified != "Error":
      # Remove existing period for consistent processing
      template = simplified.split('.')[0]

      # 5. Post-process: Map X and Y back to original variables
      if len(unique_vars) >= 2:
          # Use temporary placeholders to swap safely
          res = template.replace('X', 'TEMP_SUBJ').replace('Y', 'TEMP_PRED')
          final_output = res.replace('TEMP_SUBJ', unique_vars[0]).replace('TEMP_PRED', unique_vars[1])
      else:
          final_output = template

      # 6. Final Formatting
      final_output = final_output.strip() + ". "

      return final_output

    else:
      return "Error"

# # Example usage
# text = "It is also true that no Z are X"
# print(simplify_syllogism(text))


In [ ]:
import spacy

nlp = spacy.load("en_core_web_sm")

def lemmatize_sentence(sentence: str) -> str:
    """
    Lemmatize nouns and verbs while preserving logical structure.
    """
    doc = nlp(sentence)
    tokens = []

    for token in doc:
        # Lemmatize nouns and verbs
        if token.pos_ in {"NOUN", "PROPN", "VERB"}:
            tokens.append(token.lemma_)
        else:
            tokens.append(token.text)

    # Join tokens into a sentence (space-sensitive for punctuation)
    lemmatized_text = ""
    for i, token in enumerate(tokens):
        if i > 0 and not tokens[i].startswith(('.', ',', '?', '!', ';', ':')):
            lemmatized_text += " "
        lemmatized_text += token

    return lemmatized_text


In [ ]:
import json

# Complete end-to-end pipeline with simplification
print("\n" + "=" * 80)
print("COMPLETE END-TO-END PIPELINE: Testing Syllogisms")
print("=" * 80)

def complete_syllogism_pipeline(sentence):
    """
    Complete pipeline: LLM extraction → mapping → transformation → simplification
    Returns original, concepts, and simplified output.
    """
    original_sentence = sentence
    # Step 1: Extract concepts
    # print(f"Original sentence: {sentence}")
    lemmatized = lemmatize_sentence(sentence)
    # print(f"Lemmatized sentence: {lemmatized}")

    concepts = extract_concepts_llm(lemmatized)
    # print(f"LLM Extracted Concepts: {concepts}")

    if "error_ 1" not in concepts and "error_2" not in concepts and "error_3" not in concepts:
      # Step 2: Map concepts to placeholders
      mapping = map_concepts_to_placeholders(concepts)
      print(f"Concept→Placeholder Mapping: {mapping}")

      # Step 3: Replace concepts with placeholders
      transformed = replace_concepts_with_placeholders(lemmatized, mapping)
      print(f"Transformed (placeholders): {transformed}")

      return {
          'original': original_sentence,
          'concepts': concepts,
          'mapping': mapping,
          'simplified': transformed
      }

    else:
      return {
          'original': original_sentence,
          'concepts': concepts,
          'mapping': None,
          'simplified': None
      }


# Test on key syllogisms
# test_syllogisms = [17]

# for idx in test_syllogisms:
#     result = complete_syllogism_pipeline(syllogism_sentences[idx])

    # print(f"\n{'─'*80}")
    # print(f"Syllogism {idx}")
    # print(f"{'─'*80}")
    # print(f"Original:   {result['original']}")
    # print(f"Concepts:   {result['concepts']}")
    # print(f"Mapping:    {result['mapping']}")
    # print(f"Simplified: {result['simplified']}")

results = []

for idx, syllogism in enumerate(syllogism_sentences):
    result = complete_syllogism_pipeline(syllogism)
    result['id'] = idx
    results.append(result)
    print(f"\n{'─'*80}")
    print(f"Syllogism {idx}")
    print(f"{'─'*80}")
    print(f"Original:   {result['original']}")
    print(f"Concepts:   {result['concepts']}")
    print(f"Mapping:    {result['mapping']}")
    print(f"Simplified: {result['simplified']}")

The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.



COMPLETE END-TO-END PIPELINE: Testing Syllogisms


Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.


Output:
Resoning: Step 1: List the nouns/noun phrases in each sentence. 
               Step 2: Identify the key concepts from each sentence (Example: Aristotle is a man -> key concepts: Aristotle, man)
               Step 3: Identify which 3 concepts appear in 2 different sentences.
               Step 4: Output the results
     {
         "concepts": ["car", "vehicle", "animal"]
     }

  Note: The output is based on the given sentences. The output may change based on the input sentences. 

  Please follow the above rules and provide the output for the given sentences. 

  Sentence 1: Aristotle is a man
  Sentence 2: Aristotle is a philosopher
  Sentence 3: All men are philosophers

  OUTPUT:
     Resoning: Step 1: List the nouns/noun phrases in each sentence.
               Step 2: Identify the key concepts from each sentence (Example: Aristotle is a man -> key concepts: Aristotle, man)
               Step 3: Identify which 3 concepts appear in 2 different sentences.
               

Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.


...

Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.


Output:
Resoning: Step 1: List the nouns/noun phrases in each sentence. 
               Step 2: Identify the key concepts from each sentence (Example: Aristotle is a man -> key concepts: Aristotle, man)
               Step 3: Identify which 3 concepts appear in 2 different sentences.
               Step 4: Output the results
     {
         "concepts": ["animal", "bird", "fish"]
     } 

  Note: The output is based on the reasoning provided above. The output concepts are the ones that appear in exactly 2 different sentences. 

  Note: The output concepts are the ones that appear in exactly 2 different sentences. 

  Note: The output concepts are the ones that appear in exactly 2 different sentences. 

  Note: The output concepts are the ones that appear in exactly 2 different sentences. 

  Note: The output concepts are the ones that appear in exactly 2 different sentences. 

  Note: The output concepts are the 

  Note: The output concepts are the ones that appear in exactly 2 differe

Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.


Output:
Resoning: Step 1: List the nouns/noun phrases in each sentence. 
               Step 2: Identify the key concepts from each sentence (Example: Aristotle is a man -> key concepts: Aristotle, man)
               Step 3: Identify which 3 concepts appear in 2 different sentences.
               Step 4: Output the results
     {
         "concepts": ["dog", "poodle", "photosynthesis"]
     } 

  Note: The output is based on the reasoning that the concepts "dog", "poodle", and "photosynthesis" appear in exactly 2 different sentences. 

  Task: You will be given 3 sentences that form a syllogism. Extract the 3 core concepts from the syllogism.

  Rules:
  1. Every syllogism has 3 concepts.
  2. Each concept MUST appear in exactly 2 different sentences.
  3. Each sentence MUST contain exactly 2 concepts.
  3. NEVER include quantifiers such as: all, no, some, every in the concepts.

  Follow this step-by-step reasoning:
  Step 1: List the nouns/noun phrases in each sentence.
  Step 2: I

KeyboardInterrupt: 

In [ ]:
import json

output_path = "syllogism_results.json"

with open(output_path, "w") as f:
    json.dump(results, f, indent=4)

print(f"Results saved to {output_path}")


Results saved to syllogism_results.json


In [ ]:
results

[{'original': 'All cars are a type of vehicle. No animal is a car. Therefore, no animal can be a vehicle.',
  'concepts': ['car', 'vehicle', 'animal'],
  'mapping': {'car': 'X', 'vehicle': 'Y', 'animal': 'Z'},
  'simplified': 'All X are a type of Y. No Z is a X. Therefore, no Z can be a Y.',
  'id': 0},
 {'original': 'Nothing that is a soda is a juice. A portion of the things that are beverages are juices. The only logical conclusion is that some beverages are not sodas.',
  'concepts': ['soda', 'juice', 'beverage'],
  'mapping': {'soda': 'X', 'juice': 'Y', 'beverage': 'Z'},
  'simplified': 'Nothing that is a X is a Y. A portion of the thing that are Z are Y. The only logical conclusion is that some Z are not X.',
  'id': 1},
 {'original': 'Everything that is a planet is a celestial body. Anything that is a sun is a celestial body. There exists at least one sun that is a planet.',
  'concepts': ['planet', 'celestial body', 'sun'],
  'mapping': {'planet': 'X', 'celestial body': 'Y', 'su

In [ ]:
import json

# Complete end-to-end pipeline with simplification
print("\n" + "=" * 80)
print("COMPLETE END-TO-END PIPELINE: Testing Syllogisms")
print("=" * 80)

def complete_syllogism_pipeline(sentence):
    """
    Complete pipeline: LLM extraction → mapping → transformation → simplification
    Returns original, concepts, and simplified output.
    """
    original_sentence = sentence
    # Step 1: Extract concepts
    # print(f"Original sentence: {sentence}")
    lemmatized = lemmatize_sentence(sentence)
    # print(f"Lemmatized sentence: {lemmatized}")

    concepts = extract_concepts_llm(lemmatized)
    # print(f"LLM Extracted Concepts: {concepts}")

    if "error_ 1" not in concepts and "error_2" not in concepts and "error_3" not in concepts:
      # Step 2: Map concepts to placeholders
      mapping = map_concepts_to_placeholders(concepts)
      print(f"Concept→Placeholder Mapping: {mapping}")

      # Step 3: Replace concepts with placeholders
      transformed = replace_concepts_with_placeholders(lemmatized, mapping)
      print(f"Transformed (placeholders): {transformed}")

      # Step 4: Simplify
      simplified = ""
      for sentence in transformed.split('.')[0:3]:
        sentence += '.'
        print(f"Original sentence: {sentence}")
        simplified_sentence = simplify_syllogism(sentence)
        print(f"Simplified sentence: {simplified_sentence}")
        simplified += simplified_sentence

      return {
          'original': original_sentence,
          'concepts': concepts,
          'mapping': mapping,
          'simplified': simplified
      }

    else:
      return {
          'original': original_sentence,
          'concepts': concepts,
          'mapping': None,
          'simplified': None
      }

# Test on key syllogisms
# test_syllogisms = [17]

# for idx in test_syllogisms:
#     result = complete_syllogism_pipeline(syllogism_sentences[idx])

    # print(f"\n{'─'*80}")
    # print(f"Syllogism {idx}")
    # print(f"{'─'*80}")
    # print(f"Original:   {result['original']}")
    # print(f"Concepts:   {result['concepts']}")
    # print(f"Mapping:    {result['mapping']}")
    # print(f"Simplified: {result['simplified']}")

results = []

for idx, syllogism in enumerate(syllogism_sentences):
    if idx >= 200 and idx < 300:
      result = complete_syllogism_pipeline(syllogism)
      result['id'] =
      results.append(result)
      print(f"\n{'─'*80}")
      print(f"Syllogism {idx}")
      print(f"{'─'*80}")
      print(f"Original:   {result['original']}")
      print(f"Concepts:   {result['concepts']}")
      print(f"Mapping:    {result['mapping']}")
      print(f"Simplified: {result['simplified']}")

file_path = "syllogism_results_3.json"

with open(file_path, "w", encoding="utf-8") as f:
    # indent=4 makes the file human-readable
    json.dump(results, f, indent=4, ensure_ascii=False)

print(f"Successfully saved {len(results)} results to {file_path}")


In [ ]:
import json
import re

def parse_sentence(sentence):
    """
    Identifies the type (A, E, I, O) and the terms (Subject, Predicate).
    A: All S are P
    E: No S are P
    I: Some S are P
    O: Some S are not P
    """
    sentence = sentence.strip().rstrip('.')

    if sentence.startswith("All"):
        match = re.search(r"All (.*?) are (.*)", sentence)
        return 'A', match.group(1).strip(), match.group(2).strip()

    elif sentence.startswith("No"):
        match = re.search(r"No (.*?) are (.*)", sentence)
        return 'E', match.group(1).strip(), match.group(2).strip()

    # We check for 'O' before 'I' because 'Some...are not' contains 'Some...are'
    elif sentence.startswith("Some") and " are not " in sentence:
        match = re.search(r"Some (.*?) are not (.*)", sentence)
        return 'O', match.group(1).strip(), match.group(2).strip()

    elif sentence.startswith("Some"):
        match = re.search(r"Some (.*?) are (.*)", sentence)
        return 'I', match.group(1).strip(), match.group(2).strip()

    return None, None, None

def get_figure(p1_s, p1_p, p2_s, p2_p, c_s, c_p):
    """
    Determines the figure (1-4) based on the position of the Middle term (M).
    """
    S, P = c_s, c_p
    premise_terms = {p1_s, p1_p, p2_s, p2_p}
    m_candidates = [t for t in premise_terms if t != S and t != P]

    if not m_candidates:
        return None

    M = m_candidates[0]

    if p1_s == M and p1_p == P and p2_s == S and p2_p == M: return 1
    if p1_s == P and p1_p == M and p2_s == S and p2_p == M: return 2
    if p1_s == M and p1_p == P and p2_s == M and p2_p == S: return 3
    if p1_s == P and p1_p == M and p2_s == M and p2_p == S: return 4
    return None

def is_valid(mood, figure):
    # Updated to include all 15 unconditionally valid forms in Boolean Logic
    valid_forms = {
        1: ["AAA", "EAE", "AII", "EIO"],
        2: ["EAE", "AEE", "EIO", "AOO"],
        3: ["IAI", "AII", "OAO", "EIO"],
        4: ["AEE", "IAI", "EIO"]
    }
    return mood in valid_forms.get(figure, [])

def process_syllogisms(json_data):
    results = []
    for item in json_data:
        if not item.get("simplified"):
            continue

        sentences = [s.strip() for s in item["simplified"].split('.') if s.strip()]

        if len(sentences) != 3:
            results.append({
                "original": item["original"],
                "simplified": item["simplified"],
                "mood": None,
                "figure": None,
                "is_valid": "extraction failed"
            })
            continue

        t1, s1, p1 = parse_sentence(sentences[0])
        t2, s2, p2 = parse_sentence(sentences[1])
        tc, sc, pc = parse_sentence(sentences[2])

        if not all([t1, s1, p1, t2, s2, p2, tc, sc, pc]):
            results.append({
                "original": item["original"],
                "simplified": item["simplified"],
                "mood": None,
                "figure": None,
                "is_valid": "extraction failed"
            })
        else:
            mood = t1 + t2 + tc
            figure = get_figure(s1, p1, s2, p2, sc, pc)
            valid = is_valid(mood, figure)

            results.append({
                "original": item["original"],
                "simplified": item["simplified"],
                "mood": mood,
                "figure": figure,
                "is_valid": valid
            })
    return results

# Load and process
with open('syllogism_results_3 (1).json', 'r') as f:
    syl_data = json.load(f)

evaluated = process_syllogisms(syl_data)

In [ ]:
file_path = "evaluated_results_3_new.json"
with open(file_path, "w", encoding="utf-8") as f:
    # indent=4 makes the file human-readable
    json.dump(evaluated, f, indent=4, ensure_ascii=False)

In [ ]:
display(evaluated)

# Result Analysis

In [ ]:
import json

def compare_results(ground_truth, output_data):
    # 1. Load Ground Truth

    # 2. Create a lookup map: { "normalized_text": validity_bool }
    # We use .strip().lower() to ensure matching is robust
    gt_map = {item["syllogism"].strip().lower(): item["validity"] for item in ground_truth}

    matches = 0
    mismatches = 0
    report = []
    not_found = 0

    # 3. Iterate through your outputs
    for out in output_data:
        original_text = out["original"].strip().lower()

        if original_text in gt_map:
            actual = out["is_valid"]
            expected = gt_map[original_text]

            report.append({
                    "text": out["original"],
                    "simplified": out["simplified"],
                    "predicted": actual,
                    "ground_truth": expected,
                    "mood_fig": f"{out.get('mood')}-{out.get('figure')}"
                })

            if actual == expected:
                matches += 1
            else:
                mismatches += 1
        else:
            not_found += 1

    # 4. Print Summary Report
    total = len(output_data)
    accuracy = (matches / total) * 100 if total > 0 else 0

    print("--- Comparison Report ---")
    print(f"Total Processed: {total}")
    print(f"Correct Matches: {matches}")
    print(f"Mismatches:      {mismatches}")
    print(f"Items not in GT: {not_found}")
    print(f"Accuracy:        {accuracy:.2f}%")
    print("-------------------------\n")

    return report

with open('evaluated_results_3_new.json', 'r') as f:
    eval_data = json.load(f)
with open('train_data.json', 'r') as f:
    train_data = json.load(f)

# Path to your ground truth JSON file
report = compare_results(train_data, eval_data)

--- Comparison Report ---
Total Processed: 100
Correct Matches: 58
Mismatches:      42
Items not in GT: 0
Accuracy:        58.00%
-------------------------



In [ ]:
file_path = "report_3_new.json"
with open(file_path, "w", encoding="utf-8") as f:
    # indent=4 makes the file human-readable
    json.dump(report, f, indent=4, ensure_ascii=False)

In [ ]:
import json
with open('syllogism_results.json', 'r') as f:
    syl_data = json.load(f)

In [ ]:
with open('report_3.json', 'r') as f:
    report = json.load(f)

In [ ]:
import re
class syllogism_relation:
    def __init__(self):
        self.objects = set()
        self.inclusion_edges = dict()
        self.inversed_inclusion_edges = dict()
        self.disjunction = set()
        self.overlap = set()
        self.non_inclusion = set()

    def add_relation(self, sentence):
        keywords = sentence.split()
        if re.match(r"^All (\w+) are (\w+)$", sentence):
            self.objects.add(keywords[1])
            self.objects.add(keywords[3])
            self.inclusion_edges[keywords[3]] = keywords[1]
            self.inversed_inclusion_edges[keywords[1]] = keywords[3]
        if re.match(r"^Not all (\w+) are (\w+)$", sentence):
            self.objects.add(keywords[4])
            self.objects.add(keywords[2])
            self.non_inclusion.add(keywords[4], keywords[2])
        if re.match(r"^Some (\w+) are (\w+)$", sentence):
            self.objects.add(keywords[3])
            self.objects.add(keywords[1])
            self.overlap.add((keywords[3], keywords[1]))
        if re.match(r"^No (\w+) are (\w+)$", sentence):
            self.objects.add(keywords[3])
            self.objects.add(keywords[1])
            self.disjunction.add((keywords[3], keywords[1]))

    def conclude_relation(self, objA, objB):
        noninclusion = False
        overlap = False
        if (objA, objB) in self.disjunction or (objB, objA) in self.disjunction:
            return "disjunction"
        obj_move = objA
        while obj_move in self.inversed_inclusion_edges.keys():
            obj_pred = obj_move
            obj_move = self.inversed_inclusion_edges[obj_move]
            if (obj_move, objB) in self.non_inclusion or (objB, obj_move) in self.non_inclusion:
                noninclusion = True
            if (obj_move, objB) in self.disjunction or (objB, obj_move) in self.disjunction:
                return "disjunction"

            if obj_move == objB:
                return f"{objA} in {objB}"

        obj_move = objA
        while obj_move in self.inclusion_edges.keys():
            obj_pred = obj_move
            obj_move = self.inclusion_edges[obj_move]
            if (obj_move, objB) in self.overlap or (objB, obj_move) in self.overlap:
                overlap = True

            if obj_move == objB:
                return f"{objB} in {objA}"

        if noninclusion and overlap:
            return "nonlap"
        if noninclusion:
            return "noninclusion"
        if overlap:
            return "overlap"

        return 'unk'


In [ ]:
def verify_syllogism(syllogism):
    sentences = syllogism

In [ ]:
ground = []
pred = []
for rep in report:
    ground.append(rep['ground_truth'])
    pred.append(rep['predicted'])

In [ ]:
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(ground, pred)

In [ ]:
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [ ]:
import nltk
import string
import re

# Download necessary NLTK data if not already downloaded
try:
    nltk.data.find('tokenizers/punkt')
except nltk.downloader.DownloadError:
    nltk.download('punkt', quiet=True)

try:
    nltk.data.find('corpora/stopwords')
except nltk.downloader.DownloadError:
    nltk.download('stopwords', quiet=True)

print("NLTK 'punkt' and 'stopwords' data ensured.")

NLTK 'punkt' and 'stopwords' data ensured.


In [ ]:
import nltk
import string
import re
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

# Download necessary NLTK data if not already downloaded
nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)

print("NLTK 'punkt' and 'stopwords' data ensured.")

# Initialize an empty list for cleaned words
cleaned_words = []

# Define placeholders and stopwords
placeholders = {'x', 'y', 'z'}
english_stopwords = set(stopwords.words('english'))

# Iterate through each dictionary in the syl_data list
for item in syl_data:
    if 'simplified' in item and item['simplified'] is not None:
        sentence = item['simplified']

        # Tokenize the sentence
        tokens = word_tokenize(sentence)

        # Process each token
        for word in tokens:
            # Convert to lowercase
            word = word.lower()

            # Remove punctuation
            word = word.translate(str.maketrans('', '', string.punctuation))

            # Filter out empty strings after punctuation removal
            if not word:
                continue

            # Filter out logical placeholders and common English stopwords
            if word not in placeholders and word not in english_stopwords:
                cleaned_words.append(word)

print(f"Extracted and cleaned {len(cleaned_words)} words.")
print(f"First 20 cleaned words: {cleaned_words[:20]}")


NLTK 'punkt' and 'stopwords' data ensured.
Extracted and cleaned 1449 words.
First 20 cleaned words: ['type', 'therefore', 'nothing', 'portion', 'thing', 'logical', 'conclusion', 'everything', 'anything', 'exist', 'least', 'one', 'every', 'number', 'consequently', 'portion', 'invisible', 'select', 'classify', 'therefore']


**Reasoning**:
The previous execution failed because the `punkt_tab` NLTK resource was not found. I need to explicitly download `punkt_tab` to resolve this `LookupError`.



In [ ]:
df_syllogisms

,id,syllogism,validity,plausibility
0,50146f21-d265-4e3a-8d93-8165cdbe89a3,All cars are a type of vehicle. No animal is a...,False,True
1,dfafb4f6-4e1d-4cd5-aeb4-75d36aafdf1a,Nothing that is a soda is a juice. A portion o...,True,True
2,e30b1f83-a4c3-49cb-8aaf-5f64208c625b,Everything that is a planet is a celestial bod...,False,False
3,a30e07d5-0fb3-4097-9892-4b145b0c54f5,Every cat is an invisible creature. A number o...,True,False
4,5b8161b7-b1bf-4e16-a854-cd52cdce8a1b,There are no capital cities which are oceans. ...,True,True
...,...,...,...,...
955,6df0ab4c-8518-4efb-98d5-e1bd172b5a35,Some creatures that can fly are birds. It is t...,False,True
956,0db674a5-4391-43fd-8d40-b6ec2b6e77e3,Every single thing that is a hammer is a tool....,False,True
957,6650eef5-2045-4ee2-8fe1-a3036d54c6b6,No person who is a doctor is also a teacher. A...,True,True
958,452c2309-abd1-4cd9-9cd9-1e693ef85cf6,A creature that is a reptile is never a mammal...,True,False


In [ ]:
import nltk
import string
import re
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

# Download necessary NLTK data if not already downloaded
nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('punkt_tab', quiet=True) # Explicitly download punkt_tab

print("NLTK 'punkt' and 'stopwords' data ensured.")

# Initialize an empty list for cleaned words
cleaned_words = []

# Define placeholders and stopwords
placeholders = {'x', 'y', 'z'}
english_stopwords = set(stopwords.words('english'))

# Iterate through each dictionary in the syl_data list
for item in graph_data:
    if 'simplified' in item and item['simplified'] is not None:
        sentence = item['simplified']

        # Tokenize the sentence
        tokens = word_tokenize(sentence)

        # Process each token
        for word in tokens:
            # Convert to lowercase
            word = word.lower()

            # Remove punctuation
            word = word.translate(str.maketrans('', '', string.punctuation))

            # Filter out empty strings after punctuation removal
            if not word:
                continue

            # Filter out logical placeholders and common English stopwords
            if word not in placeholders and word not in english_stopwords:
                cleaned_words.append(word)

print(f"Extracted and cleaned {len(cleaned_words)} words.")
print(f"First 20 cleaned words: {cleaned_words[:20]}")


NLTK 'punkt' and 'stopwords' data ensured.
Extracted and cleaned 1449 words.
First 20 cleaned words: ['type', 'therefore', 'nothing', 'portion', 'thing', 'logical', 'conclusion', 'everything', 'anything', 'exist', 'least', 'one', 'every', 'number', 'consequently', 'portion', 'invisible', 'select', 'classify', 'therefore']


In [ ]:
from collections import Counter

# Calculate word frequencies
word_counts = Counter(cleaned_words)

# Print the 10 most common words
print("10 most common words and their frequencies:")
for word, count in word_counts.most_common(100):
    print(f"- {word}: {count}")

10 most common words and their frequencies:
- every: 170
- single: 118
- anything: 62
- therefore: 57
- case: 56
- thing: 47
- also: 47
- follow: 46
- true: 37
- consequently: 28
- everything: 27
- group: 25
- exist: 23
- conclude: 23
- portion: 21
- one: 20
- must: 20
- nothing: 19
- make: 19
- least: 18
- thus: 18
- number: 17
- certain: 16
- creature: 13
- set: 13
- know: 12
- mean: 12
- item: 12
- fact: 12
- imply: 12
- conclusion: 11
- without: 11
- exception: 10
- animal: 10
- consider: 10
- contain: 10
- type: 9
- live: 9
- object: 9
- people: 9
- subset: 9
- way: 8
- person: 8
- necessarily: 8
- category: 8
- logically: 8
- among: 7
- classify: 6
- result: 6
- lead: 6
- deduce: 6
- compose: 6
- class: 5
- never: 5
- piece: 5
- suggest: 5
- entire: 5
- able: 5
- logical: 4
- prove: 4
- say: 4
- grow: 4
- mutually: 4
- exclusive: 4
- use: 4
- state: 4
- call: 3
- base: 3
- belong: 3
- consequence: 3
- vehicle: 3
- describe: 3
- warm: 3
- blooded: 3
- wood: 3
- overlap: 3
- select

# Gemini Experiment

In [ ]:
import json
import pandas as pd

# Use the correct relative path from the notebook's location to the data file
with open('test_data_subtask_1.json', 'r') as f:
    data = json.load(f)

df_syllogisms = pd.DataFrame(data)

print("DataFrame loaded successfully. Displaying the first 5 rows:")
print(df_syllogisms.head())

syllogism_sentences = df_syllogisms['syllogism'].tolist()

print(f"Extracted {len(syllogism_sentences)} syllogism sentences. Displaying the first 3:")
for i, sentence in enumerate(syllogism_sentences[:3]):
    print(f"- {sentence}")

DataFrame loaded successfully. Displaying the first 5 rows:
                                     id  \
0  bff2af61-d4b0-4147-8a5b-ff4fe1892559   
1  f36a4ca3-3b69-4869-a152-deaa7e0fdad4   
2  e773bd8c-fa53-4e9c-8ec6-7d978e0601ac   
3  a2fcf47a-5df0-405d-86ce-ed8629f387a9   
4  6e5f055f-06e4-40f3-a45b-52b7d9292d60   

                                           syllogism  
0  There are no bikes that can be called cars. It...  
1  There exist some objects that are books which ...  
2  Every single object can fly. It is known that ...  
3  Every single fruit is a food. Nothing that is ...  
4  There are no individuals who are elephants tha...  
Extracted 191 syllogism sentences. Displaying the first 3:
- There are no bikes that can be called cars. It is also true that every bike is a type of vehicle. This has led to the conclusion that a portion of vehicles are bikes.
- There exist some objects that are books which are not digital files. Every single book is an item with pages. Consequentl

In [ ]:
df_syllogisms

,id,syllogism
0,34ccf2ca-3f48-4493-bd28-afa6da0c45ce,Some housecats enjoy chasing mice. Any animal ...
1,37190acf-8bb4-4c0b-995a-05937430dfad,Every pea is a vegetable. Some beans are veget...
2,94ea0032-e8dd-47bf-9a0b-49359335c390,Some hounds are actually birds. Any retriever ...
3,b0508840-0a8f-4a1f-8931-2146e18b07db,Every single car has a steering wheel. Some of...
4,9ee4c131-6dbe-436b-a116-04010e3071d1,Any animal that roars is a fish. There are no ...
...,...,...
187,440e974e-02f5-453b-8f9b-98c6d26c65f9,There are no branches that are not made of gla...
188,10bb3e13-ef4b-4ee9-9d26-f84df8b4eef7,Some office blocks are made of straw. It is tr...
189,d2fbeddb-ad92-4df0-9782-a2cf057c00d7,Every flower is a plant. All of the things tha...
190,74f5886c-08f6-4ee9-b97f-480ff3ca2007,It is a fact that all televisions are birds. A...


In [ ]:
syllogism_sentences[0].split('. ')

['There are no bikes that can be called cars',
 'It is also true that every bike is a type of vehicle',
 'This has led to the conclusion that a portion of vehicles are bikes.']

In [ ]:
def transform_syllogisms(groups_list):
    # Join groups with clear delimiters
    formatted_input = ""
    for i, group in enumerate(groups_list):
        group_split = group.split('. ')
        formatted_input += f"---\nGROUP {i+1}:\n"
        formatted_input += "\n".join(group_split) + "\n"

    return formatted_input

In [ ]:
formatted_input = transform_syllogisms(syllogism_sentences)

In [ ]:
print(formatted_input)

In [ ]:
prompt_map = f"""You are an expert in syllogism linguistics and formal logic. Your task is to translate natural language sentences into their fundamental categorical propositions.

### Task
You will receive multiple groups of sentences. For each group, provide a mapping of concepts to letters and a list of mapped sentences.

### Rules
- **Per-Group Mapping:** Reset letter assignments (A, B, C...) for every new group.
- **Concept Extraction:** Extract exactly two concepts per sentence.
- **Core Meaning:** Map different surface forms to the same concept if they are semantically identical in context (e.g., "those who breathe" and "breathers" → "Breathers").
- **Atribute Subgroups** Concepts can be changed by the presence of atributes. (eg. "gray hound" and "hound" are different concepts).
- **Symbolic Mapping:** Assign letters (A, B, C...) in order of first appearance. If a concept repeats, reuse the assigned letter.
- **Structure Preservation:** Preserve the structure of the original sentences, the only modification you must do is mapping the concepts.

### Processing Logic
1. List each sentence.
2. Identify concepts.
3. Check if either concept has appeared before.
4. Generate the final mapping and mapped sentence.

Output structure:
{{
  "results": [
    {{
      "group_id": 1,
      "mapping": {{ "concept_1": "A" , "concept_2": "B", "concept_3": "C"}},
      "mapped_sentences": ["Everything that is A is B", "There are no B that are C", "Some A are not a part of C"],
      "reasoning": "Text with procces explanation"
    }}
  ]
}}

Input Data:
{formatted_input}
---"""



In [ ]:
print(prompt)

You are an expert in syllogism linguistics and formal logic. Your task is to translate natural language sentences into their fundamental categorical propositions.

### Task
You will receive multiple groups of sentences. For each group, provide a mapping of concepts to letters and a list of simplified sentences.
The sentences should be simplified based on Aristotelian logic:
1. All X are Y (Universal Affirmative)
2. Some X are not Y (Particular Negative)
3. Some X are Y (Particular Affirmative)
4. No X are Y (Universal Negative)

### Rules
- **Per-Group Mapping:** Reset letter assignments (A, B, C...) for every new group.
- **Concept Extraction:** Extract exactly two concepts per sentence.
- **Normalization:** Strip quantifiers (all, some), negations (not, no), and copulas (is, are). You can use them to determine the simplified form.
- **Core Meaning:** Map different surface forms to the same concept if they are semantically identical in context (e.g., "those who breathe" and "breathers

In [ ]:
from google import genai

client = genai.Client(api_key='YOUR_KEY_HERE')



In [ ]:
response = client.models.generate_content(
    model="gemini-3-flash-preview",
    contents=prompt_map,
)

print(response.text)

results = response.text
dict_results = json.loads(results[8:-3])

In [ ]:
dict_results['results']

[{'group_id': 1,
  'mapping': {'Housecats': 'A',
   'Mice-chasers': 'B',
   'Cougars': 'C',
   'Predators': 'D',
   'Felines': 'E',
   'Animals': 'F',
   'Mammals': 'G',
   'Egg-layers': 'H',
   'Lions': 'I',
   'Roarers': 'J',
   'Lynx': 'K',
   'Wild cats': 'L',
   'Cats': 'M'},
  'simplified_sentences': ['Some A are B',
   'All C are D',
   'All E are F',
   'No G are H',
   'All I are J',
   'All K are L',
   'All M are F',
   'All M are E'],
  'reasoning': "Concepts like 'those who enjoy chasing mice' are normalized to 'Mice-chasers'. 'Any', 'Everything', 'All', and 'Every' indicate universal affirmatives. 'There are no' indicates a universal negative. Sentence 8 explicitly connects cats and felines."},
 {'group_id': 2,
  'mapping': {'Peas': 'A',
   'Vegetables': 'B',
   'Beans': 'C',
   'Beets': 'D',
   'Radishes': 'E',
   'Turnips': 'F',
   'Potatoes': 'G',
   'Carrots': 'H'},
  'simplified_sentences': ['All A are B',
   'Some C are B',
   'Some D are B',
   'All E are B',
   'A

In [ ]:
with open('simplified_task2.json', 'w') as f:
    json.dump(dict_results['results'],f,indent=4)

In [ ]:
with open('mapped_2.json', 'w') as f:
    json.dump(dict_results['results'],f,indent=4)

In [ ]:
def transform_mappings(mappings):
    transformed_mapping = ""
    for mapping in mappings:
         transformed_mapping += f"---\nGROUP {mapping["group_id"]}:\n"
         transformed_mapping += "\n".join(mapping["mapped_sentences"]) + "\n"

    return transformed_mapping

In [ ]:
transformed_map = transform_mappings(dict_results['results'])

In [ ]:
print(transformed_map)

In [ ]:
prompt_simplify = f"""You are an expert in syllogism linguistics and formal logic. Your task is to translate natural language sentences into their fundamental categorical propositions.

### Task
You will receive multiple groups of sentences. For each group, provide a list of simplified sentences.
The sentences should be simplified based on Aristotelian logic:
1. All X are Y (Universal Affirmative)
2. Some X are not Y (Particular Negative)
3. Some X are Y (Particular Affirmative)
4. No X are Y (Universal Negative)

### Rules
- **Normalization:** Strip quantifiers (all, some), negations (not, no), and copulas (is, are). You can use them to determine the simplified form.
- **Logical Equivalences** "Not all X are Y" is the same as "Some X are not Y" / "All X are not Y" is the same as "No X are Y" / "Only X are Y" is the same as "All Y are X".

### Processing Logic
1. List each sentence.
2. Determine the logical quantifier/relationship.
3. Generate the formal sentence.

Output structure:
{{
  "results": [
    {{
      "group_id": 1,
      "simplified_sentences": ["All A are B", "No B are C", "Some A are not C"],
      "reasoning": "Text with procces explanation"
    }}
  ]
}}

Input Data:
{transformed_map}
---"""



In [ ]:
response = client.models.generate_content(
    model="gemini-3-flash-preview",
    contents=prompt_simplify,
)

print(response.text)

results = response.text
dict_results = json.loads(results[8:-3])

```json
{
  "results": [
    {
      "group_id": 1,
      "simplified_sentences": ["No A are B", "All A are C", "Some C are A"],
      "reasoning": "Determined 'no A' as Universal Negative, 'every A' as Universal Affirmative, and 'portion of C' as Particular Affirmative."
    },
    {
      "group_id": 2,
      "simplified_sentences": ["Some A are not B", "All A are C", "Some C are B"],
      "reasoning": "'Some... not' is Particular Negative, 'Every single' is Universal Affirmative, and 'some' is Particular Affirmative."
    },
    {
      "group_id": 3,
      "simplified_sentences": ["All A are B", "Some C are not B", "Some C are not A"],
      "reasoning": "Mapped 'Every single' to All, 'some... cannot' to Some not, and 'some... are not' to Some not."
    },
    {
      "group_id": 4,
      "simplified_sentences": ["All A are B", "No C are B", "All C are A"],
      "reasoning": "'Nothing that is a C' represents a Universal Negative. 'Every single' and 'every' are Universal Affirmati

In [ ]:
with open('simple_2.json', 'w') as f:
    json.dump(dict_results['results'],f,indent=4)

In [ ]:
simple_sentences = pd.DataFrame(dict_results['results'])['simplified_sentences']

In [ ]:
#@title relation sub1

import re
class syllogism_relation:
    def __init__(self):
        self.objects = set()
        self.inclusion_edges = dict()
        self.inversed_inclusion_edges = dict()
        self.disjunction = set()
        self.overlap = set()
        self.non_inclusion = set()

    def add_relation(self, sentence):
        keywords = sentence.split()
        if re.match(r"^All (\w+) are (\w+)$", sentence):
            self.objects.add(keywords[1])
            self.objects.add(keywords[3])
            self.inclusion_edges[keywords[3]] = keywords[1]
            self.inversed_inclusion_edges[keywords[1]] = keywords[3]
        if re.match(r"^Some (\w+) are not (\w+)$", sentence):
            self.objects.add(keywords[4])
            self.objects.add(keywords[1])
            self.non_inclusion.add((keywords[1], keywords[4]))
        if re.match(r"^Some (\w+) are (\w+)$", sentence):
            self.objects.add(keywords[3])
            self.objects.add(keywords[1])
            self.overlap.add((keywords[1], keywords[3]))
        if re.match(r"^No (\w+) are (\w+)$", sentence):
            self.objects.add(keywords[3])
            self.objects.add(keywords[1])
            self.disjunction.add((keywords[1], keywords[3]))

    def conclude_relation(self, sentence):
        keywords = sentence.split()
        objA = keywords[1]
        objB = keywords[3]

        noninclusion = False
        overlap = False
        disjunction = False
        inclusion = False

        if re.match(r"^All (\w+) are (\w+)$", sentence):
            inclusion = True
        if re.match(r"^Some (\w+) are not (\w+)$", sentence):
            objB = keywords[4]
            noninclusion = True

        if re.match(r"^Some (\w+) are (\w+)$", sentence):
            overlap = True

        if re.match(r"^No (\w+) are (\w+)$", sentence):
            disjunction = True


        if (objA, objB) in self.disjunction or (objB, objA) in self.disjunction:
            if disjunction or noninclusion:
                return True

        if (objA, objB) in self.overlap or (objB, objA) in self.overlap:
            if overlap:
                return True

        if (objA, objB) in self.non_inclusion:
            if noninclusion:
                return True

        if noninclusion:
            for obj in self.objects:
                if ((objA, obj) in self.disjunction or (obj, objA) in self.disjunction) and ((objB,obj) in self.overlap or (obj, objB) in self.overlap):
                    return True
                if ((objA, obj) in self.overlap or (obj, objA) in self.overlap) and ((objB,obj) in self.disjunction or (obj, objB) in self.disjunction):
                    return True
        obj_move = objA
        while obj_move in self.inversed_inclusion_edges.keys():
            obj_move = self.inversed_inclusion_edges[obj_move]
            if (obj_move, objB) in self.disjunction or (objB, obj_move) in self.disjunction:
                if disjunction or noninclusion:
                    return True

        obj_move = objA
        while obj_move in self.inclusion_edges.keys():
            obj_move = self.inclusion_edges[obj_move]
            if (obj_move, objB) in self.non_inclusion or (obj_move,objB) in self.disjunction or (objB, obj_move) in self.disjunction:
                if noninclusion:
                    return True
            if (obj_move, objB) in self.overlap or (objB, obj_move) in self.overlap:
                if overlap:
                    return True
        obj_core = obj_move

        obj_move = objB

        while obj_move in self.inversed_inclusion_edges.keys():
            obj_move = self.inversed_inclusion_edges[obj_move]
            if (obj_move, objA) in self.disjunction or (objA, obj_move) in self.disjunction:
                if disjunction or noninclusion:
                    return True
            if (objA, obj_move) in self.non_inclusion:
                if noninclusion:
                    return True

        obj_move = objB
        while obj_move in self.inclusion_edges.keys():
            obj_move = self.inclusion_edges[obj_move]
            if (obj_move, objA) in self.overlap or (objA, obj_move) in self.overlap:
                if overlap:
                    return True
            if obj_move == objA:
                if inclusion or overlap:
                    return True

        if obj_move == obj_core:
            if overlap:
                return True

        # print(objB,self.disjunction, self.inclusion_edges)
        return False


In [ ]:
import re
class syllogism_relation:
    def __init__(self):
        self.objects = dict()
        self.inclusion_edges = dict()
        self.inversed_inclusion_edges = dict()
        self.disjunction = dict()
        self.overlap = dict()
        self.non_inclusion = dict()

    def add_relation(self, sentence, id):
        keywords = sentence.split()
        if re.match(r"^All (\w+) are (\w+)$", sentence):
            self.objects.add(keywords[1])
            self.objects.add(keywords[3])
            self.inclusion_edges[keywords[3]] = (keywords[1],id)
            self.inversed_inclusion_edges[keywords[1]] = (keywords[3],i)
        if re.match(r"^Some (\w+) are not (\w+)$", sentence):
            self.objects.add(keywords[4])
            self.objects.add(keywords[1])
            self.non_inclusion[id]=(keywords[1], keywords[4])
        if re.match(r"^Some (\w+) are (\w+)$", sentence):
            self.objects.add(keywords[3])
            self.objects.add(keywords[1])
            self.overlap[id]=(keywords[1], keywords[3])
        if re.match(r"^No (\w+) are (\w+)$", sentence):
            self.objects.add(keywords[3])
            self.objects.add(keywords[1])
            self.disjunction[id]=(keywords[1], keywords[3])

    def conclude_relation(self, sentence):
        keywords = sentence.split()
        objA = keywords[1]
        objB = keywords[3]

        noninclusion = False
        overlap = False
        disjunction = False
        inclusion = False

        if re.match(r"^All (\w+) are (\w+)$", sentence):
            inclusion = True
        if re.match(r"^Some (\w+) are not (\w+)$", sentence):
            objB = keywords[4]
            noninclusion = True

        if re.match(r"^Some (\w+) are (\w+)$", sentence):
            overlap = True

        if re.match(r"^No (\w+) are (\w+)$", sentence):
            disjunction = True


        if (objA, objB) in self.disjunction.values() or (objB, objA) in self.disjunction.values():
            if disjunction or noninclusion:
                return True

        if (objA, objB) in self.overlap or (objB, objA) in self.overlap:
            if overlap:
                return True

        if (objA, objB) in self.non_inclusion:
            if noninclusion:
                return True

        if noninclusion:
            for obj in self.objects:
                if ((objA, obj) in self.disjunction or (obj, objA) in self.disjunction) and ((objB,obj) in self.overlap or (obj, objB) in self.overlap):
                    return True
                if ((objA, obj) in self.overlap or (obj, objA) in self.overlap) and ((objB,obj) in self.disjunction or (obj, objB) in self.disjunction):
                    return True
        obj_move = objA
        while obj_move in self.inversed_inclusion_edges.keys():
            obj_move = self.inversed_inclusion_edges[obj_move]
            if (obj_move, objB) in self.disjunction or (objB, obj_move) in self.disjunction:
                if disjunction or noninclusion:
                    return True

        obj_move = objA
        while obj_move in self.inclusion_edges.keys():
            obj_move = self.inclusion_edges[obj_move]
            if (obj_move, objB) in self.non_inclusion or (obj_move,objB) in self.disjunction or (objB, obj_move) in self.disjunction:
                if noninclusion:
                    return True
            if (obj_move, objB) in self.overlap or (objB, obj_move) in self.overlap:
                if overlap:
                    return True
        obj_core = obj_move

        obj_move = objB

        while obj_move in self.inversed_inclusion_edges.keys():
            obj_move = self.inversed_inclusion_edges[obj_move]
            if (obj_move, objA) in self.disjunction or (objA, obj_move) in self.disjunction:
                if disjunction or noninclusion:
                    return True
            if (objA, obj_move) in self.non_inclusion:
                if noninclusion:
                    return True

        obj_move = objB
        while obj_move in self.inclusion_edges.keys():
            obj_move = self.inclusion_edges[obj_move]
            if (obj_move, objA) in self.overlap or (objA, obj_move) in self.overlap:
                if overlap:
                    return True
            if obj_move == objA:
                if inclusion or overlap:
                    return True

        if obj_move == obj_core:
            if overlap:
                return True

        # print(objB,self.disjunction, self.inclusion_edges)
        return False


In [ ]:
ex1=['All A are B', 'All C are A', 'All C are B'] #false
ex2=['No A are B', 'All A are C', 'Some C are not B'] #false

In [ ]:
simple_sentences[65]

['All A are B', 'No B are C', 'Some C are not A']

In [ ]:
def get_validity(sentences):
    syll = syllogism_relation()
    for sentence in sentences[:-1]:
        syll.add_relation(sentence)
    return syll.conclude_relation(sentences[-1])



In [ ]:
exam = ["All A are B", "All A are C", "Some B are C"]

In [ ]:
get_validity(ex1)

True

In [ ]:
validities = []
for simp in simple_sentences:
    validities.append(get_validity(simp))

In [ ]:
validities1 == validities

True

In [ ]:
len(validities)

360

In [ ]:
ids =df_syllogisms['id'].to_list()

NameError: name 'df_syllogisms' is not defined

In [ ]:
with open('predictions_5.json', 'r') as f:
    pred4 = json.load(f)

In [ ]:
pred4

[{'id': 'bff2af61-d4b0-4147-8a5b-ff4fe1892559', 'validity': True},
 {'id': 'f36a4ca3-3b69-4869-a152-deaa7e0fdad4', 'validity': False},
 {'id': 'e773bd8c-fa53-4e9c-8ec6-7d978e0601ac', 'validity': True},
 {'id': 'a2fcf47a-5df0-405d-86ce-ed8629f387a9', 'validity': False},
 {'id': '6e5f055f-06e4-40f3-a45b-52b7d9292d60', 'validity': True},
 {'id': '80a62ff2-1bc5-48e6-bc9f-f877690852fc', 'validity': False},
 {'id': 'b73d8e0e-782e-4e60-866b-595cf13a3421', 'validity': False},
 {'id': 'fd552276-66bf-45da-8cea-67192fe99fc3', 'validity': True},
 {'id': '8041eb99-e73f-46ca-a2d7-9aefd2c6c0b1', 'validity': True},
 {'id': 'ad09c938-3d1d-4d5e-b75f-04fb0098ba43', 'validity': False},
 {'id': 'af77d792-0970-4650-8ccf-4218f7685b37', 'validity': False},
 {'id': '9ee7aff6-1237-4e93-b3e4-08434de41ede', 'validity': False},
 {'id': '552b33b2-6a6a-4a10-8a76-5d0d7c3d35ff', 'validity': False},
 {'id': '41ee96a0-4400-4ba7-a4b3-54d49b5942cc', 'validity': True},
 {'id': 'f682283e-eb75-47d9-9e48-ca918d7c3737', 'valid

In [ ]:
competition_results = []
for id, val in zip(ids,validities):
    competition_results.append({'id':id,'validity':val})

In [ ]:
competition_results2 == competition_results

True

In [ ]:
with open('final_predictions.json', 'w') as f:
    json.dump(competition_results,f, indent=4)

In [ ]:
pd.DataFrame(pred4)['validity'].to_list()== validities

True

In [ ]:
for i in range(len(validities)):
    if validities[i] != pred4[i]['validity']:
        print(df_syllogisms['syllogism'][i])
        print(simp_results[i]['simplified_sentences'])
        print(validities[i])
        print(pred4[i]['validity'])

In [ ]:
for ix, syls in enumerate(simp_results):
    print(syls['simplified_sentences'])
    print(df_syllogisms['syllogism'][ix])
    print()

['No A are B', 'All A are C', 'Some C are A']
There are no bikes that can be called cars. It is also true that every bike is a type of vehicle. This has led to the conclusion that a portion of vehicles are bikes.

['Some A are not B', 'All A are C', 'Some C are B']
There exist some objects that are books which are not digital files. Every single book is an item with pages. Consequently, some items with pages are digital files.

['All A are B', 'Some C are not B', 'Some C are not A']
Every single object can fly. It is known that some boats cannot fly. It follows that some boats are not objects.

['All A are B', 'No C are B', 'All C are A']
Every single fruit is a food. Nothing that is a vegetable is a food. It follows from this that every vegetable is a fruit.

['No A are B', 'All C are B', 'Some C are not A']
There are no individuals who are elephants that are also people enrolled in kindergarten. All five-year-olds are people enrolled in kindergarten. It can be deduced that some five-

In [ ]:
import math
import sys
from typing import List, Dict, Any, Tuple, Optional

# --- 1. CORE FUNCTIONS FOR ACCURACY AND BIAS CALCULATION ---

def calculate_accuracy(
    ground_truth_list: List[Dict[str, Any]],
    predictions_list: List[Dict[str, Any]],
    metric_name: str,
    prediction_key: str,
    plausibility_filter: Optional[bool] = None
) -> Tuple[float, int, int]:
    """
    Calculates the accuracy of 'validity' predictions against ground truth labels,
    with an optional filter based on ground truth 'plausibility'.
    """
    gt_map = {item['id']: item for item in ground_truth_list}

    correct_predictions = 0
    total_predictions = 0

    for pred_item in predictions_list:
        item_id = pred_item['id']

        if item_id in gt_map:
            gt_item = gt_map[item_id]

            gt_plausibility = gt_item.get('plausibility')
            if plausibility_filter is not None and gt_plausibility != plausibility_filter:
                continue

            if metric_name in gt_item and prediction_key in pred_item:
                true_label = gt_item[metric_name]
                predicted_label = pred_item[prediction_key]

                if isinstance(true_label, bool) and isinstance(predicted_label, bool):
                    total_predictions += 1
                    if true_label == predicted_label:
                        correct_predictions += 1

    if total_predictions == 0:
        return 0.0, 0, 0

    accuracy = (correct_predictions / total_predictions) * 100
    return accuracy, correct_predictions, total_predictions


def calculate_subgroup_accuracy(
    gt_map: Dict[str, Any],
    predictions_list: List[Dict[str, Any]],
    gt_validity: bool,
    gt_plausibility: bool
) -> Tuple[float, int, int]:
    """
    Calculates accuracy for a specific subgroup defined by ground truth validity AND plausibility.
    """
    correct_predictions = 0
    total_predictions = 0

    for pred_item in predictions_list:
        item_id = pred_item['id']
        if item_id in gt_map:
            gt_item = gt_map[item_id]

            if gt_item.get('validity') == gt_validity and gt_item.get('plausibility') == gt_plausibility:
                if 'validity' in gt_item and 'validity' in pred_item:
                    true_label = gt_item['validity']
                    predicted_label = pred_item['validity']

                    if isinstance(true_label, bool) and isinstance(predicted_label, bool):
                        total_predictions += 1
                        if true_label == predicted_label:
                            correct_predictions += 1

    if total_predictions == 0:
        return 0.0, 0, 0

    accuracy = (correct_predictions / total_predictions) * 100
    return accuracy, correct_predictions, total_predictions


def calculate_content_effect_bias(accuracies: Dict[str, float]) -> Dict[str, float]:
    """
    Calculates the content effect bias metrics as defined.
    """

    acc_plausible_valid = accuracies.get('acc_plausible_valid', 0.0)
    acc_implausible_valid = accuracies.get('acc_implausible_valid', 0.0)
    acc_plausible_invalid = accuracies.get('acc_plausible_invalid', 0.0)
    acc_implausible_invalid = accuracies.get('acc_implausible_invalid', 0.0)

    # 1. content_effect_intra_validity_label
    intra_valid_diff = abs(acc_plausible_valid - acc_implausible_valid)
    intra_invalid_diff = abs(acc_plausible_invalid - acc_implausible_invalid)
    content_effect_intra_validity_label = (intra_valid_diff + intra_invalid_diff) / 2.0

    # 2. content_effect_inter_validity_label
    inter_plausible_diff = abs(acc_plausible_valid - acc_plausible_invalid)
    inter_implausible_diff = abs(acc_implausible_valid - acc_implausible_invalid)
    content_effect_inter_validity_label = (inter_plausible_diff + inter_implausible_diff) / 2.0

    # 3. tot_content_effect
    tot_content_effect = (content_effect_intra_validity_label + content_effect_inter_validity_label) / 2.0

    return {
        'content_effect_intra_validity_label': content_effect_intra_validity_label,
        'content_effect_inter_validity_label': content_effect_inter_validity_label,
        'tot_content_effect': tot_content_effect
    }

def calculate_smooth_combined_metric(overall_accuracy: float, total_content_effect: float) -> float:
    """
    Computes a smooth combined score using the natural logarithm.
    Formula: accuracy / (1 + ln(1 + content_effect))
    """
    if total_content_effect < 0:
        return 0.0

    log_penalty = math.log(1 + total_content_effect)
    combined_smooth_score = overall_accuracy / (1 + log_penalty)
    return combined_smooth_score


def run_full_scoring(reference_data_path: str, prediction_path: str, output_path: str):
    """
    Runs the full analysis pipeline for a single submission and writes results to the output path.
    """

    try:
        # 1. Load data from the provided file paths
        with open(reference_data_path, 'r') as f:
            ground_truth = json.load(f)

        with open(prediction_path, 'r') as f:
            predictions = json.load(f)

        # Check that the examples in ground_truth are all covered in predictions
        gt_ids = set([example["id"] for example in ground_truth])
        pd_ids = set([example["id"] for example in predictions])
        diff = len(gt_ids.difference(pd_ids))

        if diff != 0:
            print(f"Error: not all the examples in ground truth have a corresponding prediction", file=sys.stderr)
            final_results = {'accuracy': 0.0, 'f1_premises': 0.0, 'combined_score': 0.0}
            # --- Write Results ---
            try:
                with open(output_path, 'w') as f:
                    json.dump(final_results, f)
                    print(f"Scoring complete. Results written to {output_path}")
            except Exception as e:
                print(f"Error writing final results to file: {e}", file=sys.stderr)

            return

    except FileNotFoundError as e:
        print(f"Error: Required file not found. {e}", file=sys.stderr)
        final_results = {'accuracy': 0.0, 'content_effect': 100.0, 'combined_score': 0.0}
    except json.JSONDecodeError as e:
        print(f"Error: Invalid JSON format in submission or reference file. {e}", file=sys.stderr)
        final_results = {'accuracy': 0.0, 'content_effect': 100.0, 'combined_score': 0.0}
    except Exception as e:
        print(f"An unexpected error occurred during file loading: {e}", file=sys.stderr)
        final_results = {'accuracy': 0.0, 'content_effect': 100.0, 'combined_score': 0.0}

    else:
        gt_map = {item['id']: item for item in ground_truth}

        # --- 2. Calculate Overall Accuracy ---
        common_args = {
            'ground_truth_list': ground_truth,
            'predictions_list': predictions,
            'metric_name': 'validity',
            'prediction_key': 'validity'
        }
        overall_acc, _, _ = calculate_accuracy(**common_args, plausibility_filter=None)

        # --- 3. Calculate Conditional Accuracies for Bias Metrics ---
        acc_plausible_valid, _, _ = calculate_subgroup_accuracy(gt_map, predictions, gt_validity=True, gt_plausibility=True)
        acc_implausible_valid, _, _ = calculate_subgroup_accuracy(gt_map, predictions, gt_validity=True, gt_plausibility=False)
        acc_plausible_invalid, _, _ = calculate_subgroup_accuracy(gt_map, predictions, gt_validity=False, gt_plausibility=True)
        acc_implausible_invalid, _, _ = calculate_subgroup_accuracy(gt_map, predictions, gt_validity=False, gt_plausibility=False)

        conditional_accuracies = {
            'acc_plausible_valid': acc_plausible_valid,
            'acc_implausible_valid': acc_implausible_valid,
            'acc_plausible_invalid': acc_plausible_invalid,
            'acc_implausible_invalid': acc_implausible_invalid
        }

        # --- 4. Calculate Content Effect Bias Metrics ---
        bias_metrics = calculate_content_effect_bias(conditional_accuracies)
        tot_content_effect = bias_metrics['tot_content_effect']

        # --- 5. Calculate Combined Metric ---
        combined_smooth_score = calculate_smooth_combined_metric(overall_acc, tot_content_effect)

        # --- 6. Prepare Final Output Dictionary ---
        final_results = {
            'accuracy': round(overall_acc, 4),
            'content_effect': round(tot_content_effect, 4),
            'combined_score': round(combined_smooth_score, 4)
        }

    # --- 7. Write Results ---
    try:
        with open(output_path, 'w') as f:
            json.dump(final_results, f)
            print(f"Scoring complete. Results written to {output_path}")
    except Exception as e:
        print(f"Error writing final results to file: {e}", file=sys.stderr)


In [ ]:
run_full_scoring("train_data_200.json", "train_predictions.json", "rezz.json")

Scoring complete. Results written to rezz.json


In [ ]:
ground_truth = df_syllogisms['validity'].to_list()

In [ ]:
len(validities)

200

In [ ]:
gt = ground_truth[600:]

In [ ]:
from sklearn.metrics import accuracy_score, confusion_matrix
accuracy_score(gt, validities)

0.95

In [ ]:
cm = confusion_matrix(gt, validities)
cm

array([[95,  8],
       [11, 86]])

In [ ]:
for i in range(200):
    if gt[i] != validities[i]:
        print(simple_sentences[i])
        print(syllogism_sentences[600+i])
        print('Ground: ',gt[i])
        print('Pred: ',validities[i])

['All A are B', 'All C are A', 'All C are B']
Every single animal is a living thing. Anything that is a bird is an animal. All birds are living things.
Ground:  False
Pred:  True
['All A are B', 'No C are A', 'Some C are not B']
Anything that is a dog is a mammal. There are no sharks which are also dogs. It is the case that some sharks are not mammals.
Ground:  True
Pred:  False
['No A are B', 'Some C are B', 'Some C are not A']
There are no animals that are also plants. At least one tree is a plant. At least one tree is not an animal.
Ground:  False
Pred:  True
['All A are B', 'All A are C', 'Some C are B']
Anything that is a spider is a mammal. All spiders are insects. This means that a few insects are mammals.
Ground:  False
Pred:  True
['All A are B', 'All C are B', 'All C are A']
Every object that orbits a star is a planet. Anything that is a gas giant is an object that is a planet. Therefore, there are no gas giants that do not orbit a star.
Ground:  True
Pred:  False
['All A are

In [ ]:
data_trunc = data[:200]


with open("train_data_200.json", 'w') as f:
    json.dump(data_trunc,f, indent=4)
